# here we will create gold tables and using constellelation schema with multiple fact tables -- the percentage utilization rate and shrinkage % is calculated in power BI using DAX queries

In [0]:
# Quick Schema Check: Print exact Bronze column names
fact_tables = [
    "fact_shift", 
    "fact_work", 
    "fact_shrinkage"
]

for tbl in fact_tables:
    print(f"=== Columns in 7trents_data_platform.bronze.{tbl} ===")
    spark.table(f"7trents_data_platform.bronze.{tbl}").printSchema()

In [0]:
# Quick Schema Check: Print exact Bronze column names
dimension_tables = [
    "dim_people", 
    "dim_county", 
    "dim_cancellation_reason", 
    "dim_shrinkage_type"
]

for tbl in dimension_tables:
    print(f"=== Columns in 7trents_data_platform.bronze.{tbl} ===")
    spark.table(f"7trents_data_platform.bronze.{tbl}").printSchema()

In [0]:
# ==============================================================================
# NOTEBOOK: 01_gold_dimensions
# OBJECTIVE: Generate Calendar Dimension & Materialize Gold Dimension Tables
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DateType, BooleanType

silver_schema = "7trents_data_platform.silver"
gold_schema = "7trents_data_platform.gold"

# Ensure Gold database exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Reading Silver from: {silver_schema}")
print(f"Writing Gold to:     {gold_schema}")


In [0]:


# ------------------------------------------------------------------------------
# 1. GOLD.DIM_DATE (Standard Calendar Dimension for Power BI Time Intelligence)
# ------------------------------------------------------------------------------
print("\n--- Generating: gold.dim_date ---")
start_date = "2026-04-01"
end_date = "2026-04-30"

df_date = (
    spark.sql(f"""
        SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) as calendar_date
    """)
    .withColumn("date_key", F.date_format(F.col("calendar_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("day_of_month", F.dayofmonth(F.col("calendar_date")).cast(IntegerType()))
    .withColumn("day_name", F.date_format(F.col("calendar_date"), "EEEE"))
    .withColumn("day_of_week", F.dayofweek(F.col("calendar_date")).cast(IntegerType()))
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), True).otherwise(False))
    .withColumn("week_of_year", F.weekofyear(F.col("calendar_date")).cast(IntegerType()))
    .withColumn("month_number", F.month(F.col("calendar_date")).cast(IntegerType()))
    .withColumn("month_name", F.date_format(F.col("calendar_date"), "MMMM"))
    .withColumn("quarter", F.quarter(F.col("calendar_date")).cast(IntegerType()))
    .withColumn("year", F.year(F.col("calendar_date")).cast(IntegerType()))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "date_key",
        "calendar_date",
        "day_of_month",
        "day_name",
        "day_of_week",
        "is_weekend",
        "week_of_year",
        "month_number",
        "month_name",
        "quarter",
        "year",
        "_gold_processed_at"
    )
)

(
    df_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_date")
)
print(f"✓ Saved {gold_schema}.dim_date: {df_date.count()} rows")


In [0]:


# ------------------------------------------------------------------------------
# 2. GOLD.DIM_PEOPLE (Staff Profiles)
# ------------------------------------------------------------------------------
print("\n--- Materializing: gold.dim_people ---")
df_silver_people = spark.table(f"{silver_schema}.dim_people")

df_gold_people = (
    df_silver_people
    .withColumn("people_id", F.col("people_id").cast(IntegerType()))
    .withColumn("hire_date", F.to_date(F.col("hire_date")))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "people_id",
        "employee_name",
        "first_name",
        "surname",
        "job_role",
        "employment_type",
        "team",
        "hire_date",
        "_gold_processed_at"
    )
)

(
    df_gold_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_people")
)
print(f"✓ Saved {gold_schema}.dim_people: {df_gold_people.count()} rows")



In [0]:

# ------------------------------------------------------------------------------
# 3. GOLD.DIM_COUNTY (Regional Hubs)
# ------------------------------------------------------------------------------
print("\n--- Materializing: gold.dim_county ---")
df_silver_county = spark.table(f"{silver_schema}.dim_county")

df_gold_county = (
    df_silver_county
    .withColumn("county_id", F.col("county_id").cast(IntegerType()))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "county_id",
        "county_name",
        "region",
        "country",
        "population_band",
        "urban_rural",
        "_gold_processed_at"
    )
)

(
    df_gold_county.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_county")
)
print(f"✓ Saved {gold_schema}.dim_county: {df_gold_county.count()} rows")


In [0]:


# ------------------------------------------------------------------------------
# 4. GOLD.DIM_CANCELLATION_REASON (Work Order Cancellation Codes)
# ------------------------------------------------------------------------------
print("\n--- Materializing: gold.dim_cancellation_reason ---")
df_silver_cancel = spark.table(f"{silver_schema}.dim_cancellation_reason")

df_gold_cancel = (
    df_silver_cancel
    .withColumn("cancellation_id", F.col("cancellation_id").cast(IntegerType()))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "cancellation_id",
        "cancellation_reason",
        "reason_group",
        "is_customer_driven",
        "severity",
        "_gold_processed_at"
    )
)

(
    df_gold_cancel.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_cancellation_reason")
)
print(f"✓ Saved {gold_schema}.dim_cancellation_reason: {df_gold_cancel.count()} rows")


In [0]:


# ------------------------------------------------------------------------------
# 5. GOLD.DIM_SHRINKAGE_TYPE (Absence & Out-of-Office Codes)
# ------------------------------------------------------------------------------
print("\n--- Materializing: gold.dim_shrinkage_type ---")
df_silver_shrinkage = spark.table(f"{silver_schema}.dim_shrinkage_type")

df_gold_shrinkage_type = (
    df_silver_shrinkage
    .withColumn("shrinkage_type_id", F.col("shrinkage_type_id").cast(IntegerType()))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "shrinkage_type_id",
        "shrinkage_type",
        "category",
        "is_paid",
        "is_planned",
        "_gold_processed_at"
    )
)

(
    df_gold_shrinkage_type.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_shrinkage_type")
)
print(f"✓ Saved {gold_schema}.dim_shrinkage_type: {df_gold_shrinkage_type.count()} rows")



In [0]:

# ------------------------------------------------------------------------------
# 6. VERIFICATION AUDIT
# ------------------------------------------------------------------------------
print("\n" + "="*50)
print("GOLD DIMENSIONS STATUS:")
print("="*50)
for tbl in ["dim_date", "dim_people", "dim_county", "dim_cancellation_reason", "dim_shrinkage_type"]:
    print(f"Table: {gold_schema}.{tbl:<28} Rows: {spark.table(f'{gold_schema}.{tbl}').count()}")


# this code works perfectly fine - it also works good when job and pipeline is done